# Proyecto #1: Biodiversity at Scale
## Parte 2: Baseline con Multilayer Perceptron (MLP)
**Maestría de Investigación en IA - UTEC Posgrado**  
**Curso**: Aprendizaje Profundo – Práctica  
**Profesor**: Dra. Aurea Soriano-Vargas

---
### Objetivos de esta etapa:
1. Implementar y justificar un modelo base completamente conectado (**MLP**) para clasificación de especies.
2. Entrenar y evaluar el modelo reportando: **Training Loss, Validation Loss, Training Accuracy, Validation Accuracy, Macro F1 y tiempo de cómputo**.
3. Demostrar experimental y teóricamente las limitaciones intrínsecas de utilizar arquitecturas densas sobre datos bidimensionales de imagen.


In [ ]:
import sys
from pathlib import Path
ROOT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(ROOT_DIR))

import torch
from src.utils.seed import seed_everything
from src.data.dataset import SyntheticINatDataset
from src.data.dataloader import build_dataloaders
from src.models.factory import build_model, count_parameters
from src.training.losses import build_criterion
from src.training.optimizers import build_optimizer
from src.training.trainer import Trainer
from src.utils.tracking import ExperimentTracker
from src.utils.visualization import plot_training_curves

SEED = 42
seed_everything(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tracker = ExperimentTracker(log_dir=str(ROOT_DIR / "logs"))
print(f"Device: {device}")


### 1. Justificación de Diseño del MLP
Para este baseline, redimensionamos las imágenes a una resolución compacta de **$32 \times 32$ píxeles**:
- **Dimensión de entrada**: $3 \times 32 \times 32 = 3,072$ valores continuos.
- **Capas ocultas**: $[512, 256]$ neuronas con activación **ReLU** para evitar desvanecimiento de gradiente.
- **Inicialización de pesos**: He / Kaiming Normal (`kaiming_normal_`), ideal para activaciones ReLU.
- **Optimizador**: Adam ($lr = 10^{-3}$) por su rápida convergencia empírica en modelos densos.
- **Función de pérdida**: Cross-Entropy Loss estándar.


In [ ]:
N_CLASSES = 50
BATCH_SIZE = 32
EPOCHS = 5

# Para el prototipado inicial usamos dataset sintético / representativo
train_ds = SyntheticINatDataset(num_samples=50 * 35, num_classes=N_CLASSES, img_size=32, seed=SEED)
val_ds = SyntheticINatDataset(num_samples=50 * 10, num_classes=N_CLASSES, img_size=32, seed=SEED+1)

train_loader, val_loader = build_dataloaders(train_ds, val_ds, batch_size=BATCH_SIZE, num_workers=2, seed=SEED)

# Instanciación del MLP Baseline
mlp_model = build_model(
    "mlp",
    num_classes=N_CLASSES,
    input_shape=(3, 32, 32),
    hidden_dims=[512, 256],
    activation="relu",
    dropout=0.1
)

total_p, train_p, total_m, train_m = count_parameters(mlp_model)
print(f"Parámetros del MLP: Total={total_p:,} ({total_m:.2f} M) | Entrenables={train_p:,}")


### 2. Entrenamiento y Evaluación del MLP (Experimento E1)


In [ ]:
criterion = build_criterion("cross_entropy")
optimizer = build_optimizer(mlp_model, opt_type="adam", lr=1e-3, weight_decay=1e-4)

trainer = Trainer(
    model=mlp_model,
    criterion=criterion,
    optimizer=optimizer,
    device=device,
    use_amp=True
)

history, best_metrics, peak_vram, total_time = trainer.fit(
    train_loader, val_loader, epochs=EPOCHS, verbose=True
)

# Registro del Experimento E1 en el Tracker oficial
tracker.log_experiment(
    exp_id="E1",
    model_name="MLP Baseline (32x32)",
    optimizer="Adam (lr=1e-3)",
    regularization="None",
    augmentation="None",
    transfer_learning="No",
    long_tail="No",
    metrics=best_metrics,
    training_time_sec=total_time,
    peak_vram_mb=peak_vram,
    param_count_m=total_m
)


### 3. Curvas de Aprendizaje del MLP


In [ ]:
plot_training_curves(history, title="Experimento E1: MLP Baseline Learning Curves")


### 4. Discusión Teórica y Experimental: Limitaciones del MLP
A partir de los resultados obtenidos, documentamos las respuestas a las preguntas clave de la rúbrica:

1. **¿Qué información espacial se pierde al convertir una imagen en un vector?**  
   Al aplanar una matriz de $C \times H \times W$ a un vector 1D, se rompe la **topología bidimensional** y la localidad espacial. Para el MLP, el píxel $(i, j)$ está matemáticamente tan desconectado de su vecino $(i, j+1)$ como del píxel en la esquina opuesta $(H, W)$. Se destruyen las correlaciones locales de bordes, texturas y formas contiguas.

2. **¿Qué limitaciones tiene un MLP frente a una CNN?**  
   - **Falta de invarianza y equivariancia traslacional**: Una flor o ave ubicada en la esquina superior izquierda activa neuronas completamente distintas a las de la misma ave centrada.
   - **Ausencia de compartición de pesos (*weight sharing*)**: Cada conexión tiene su propio peso independiente, lo que incrementa masivamente la propensión al sobreajuste (*overfitting*).

3. **¿Cómo afecta el tamaño de la entrada al número de parámetros? (Explosión Paramétrica)**:
   - Para $32 \times 32 \times 3$: $3,072 \times 512 \approx 1.57$ millones de pesos solo en la primera capa.
   - Para $96 \times 96 \times 3$: $27,648 \times 512 \approx 14.15$ millones de pesos.
   - Para la resolución estándar de $224 \times 224 \times 3$: $150,528 \times 512 \approx \mathbf{77.07}$ **millones de pesos** en una sola capa lineal, volviéndose computacionalmente intratable e ineficiente.
